In [6]:
import numpy as np
from tqdm import tqdm
import os
import json
from PIL import Image
import cv2
import torch
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import DataLoader
from xiaoying import net
from xiaoying.validation import evaluate_utils, evaluate
from xiaoying.expert.utils import get_expert_features, combined_features
from xiaoying.expert.hog import get_hog_features
from xiaoying.data import CustomImageFolderDataset, val_dataset
from xiaoying.validation_mixed import validate_IJB_BC
from xiaoying.ViT.load_models import load_vit_base

In [4]:
val_ds = val_dataset(data_root='datasets/data')
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False, drop_last=True)

loading validation data memfile
loading validation data memfile
loading validation data memfile
loading validation data memfile


In [3]:
features = get_hog_features(val_loader)

100%|██████████| 101/101 [00:23<00:00,  4.29it/s]


In [5]:
torch.save(features, 'features_hog.pt')

In [11]:
device = torch.device('cuda')
print(device)

model_name = 'ir_50'
checkpoint = 'models\ir_50.pth'
state_dict = torch.load(checkpoint)
model = net.build_model(model_name)
model.load_state_dict(state_dict)

# vit = load_vit_base()
# vit.load_state_dict(torch.load('C:\MINH\models\ViT_base.pth'))

# class ViT(nn.Module):
#     def __init__(self, model):
#         super(ViT, self).__init__()
#         self.model = model
        
#     def forward(self, x):
#         x = self.model(x)
#         norm = torch.norm(x, 2, 1, True)
#         output = torch.div(x, norm)
#         return output, norm

# model = ViT(vit)
model.to(device);

cuda


In [5]:
evaluate.evaluate1(model, val_loader, device, '', 'features_hog.pt')

100%|██████████| 101/101 [11:05<00:00,  6.59s/it]


	[agedb_30] Acc=0.6497, Th=1.4940
	[cfp_fp] Acc=0.5000, Th=1.8000
	[lfw] Acc=0.8273, Th=1.5360
	[cfp_ff] Acc=0.4905, Th=3.5930


np.float64(0.6168796524866463)

In [15]:
validate_IJB_BC.USE_HOG_EXPERT = True
validate_IJB_BC.EXPERT_FEATS_PATH = 'ijbc_hog.pt'
validate_IJB_BC.FEATS_PATH = 'ijbc.pt'

In [16]:
model.eval()
r = evaluate.evaluate2(r'datasets\ijb-testsuite\ijb', model, 'IR', 'IJBC', 512, device=device)

result save_path ./result/IJBC/IR
files: 469375
total images : 469375
Done loaded Hog features!
Create features...


100%|██████████| 917/917 [53:40<00:00,  3.51s/it] 


Combined features!
Feature Shape: (469375 , 1024) .
(5832,) (5832,)
(6024,) (6024,)
(11856,) (11856,)
total_templates (469375,) (469375,)
Finish Calculating 0 template features.
Finish Calculating 2000 template features.
gallery_templates_feature (3531, 1024)
gallery_unique_subject_ids (3531,)
(457519,) (457519,)
Finish Calculating 0 template features.
Finish Calculating 2000 template features.
Finish Calculating 4000 template features.
Finish Calculating 6000 template features.
Finish Calculating 8000 template features.
Finish Calculating 10000 template features.
Finish Calculating 12000 template features.
Finish Calculating 14000 template features.
Finish Calculating 16000 template features.
Finish Calculating 18000 template features.
probe_mixed_templates_feature (19593, 1024)
probe_mixed_unique_subject_ids (19593,)
(19593, 1024)
(3531, 1024)
similarity shape (19593, 3531)
(19593, 3531)
top1 = 0.03654366355330985
top5 = 0.059102740774766495
top10 = 0.07400602255907722
69163290
(1959

In [6]:
from sklearn.decomposition import PCA

pca = PCA(n_components=512)
X_reduced = pca.fit_transform(f.numpy())

print(X_reduced.shape)

(469375, 512)
